# Streamlit 앱 Colab 테스트

## Goal

GitHub의 최신 `streamlit_app`을 Colab에 내려받아 의존성, 단위 테스트, 전처리 함수, Streamlit 화면을 순서대로 검증합니다. 기존 Colab 분석 프로그램은 수정하지 않습니다.

> 실행 전 로컬 변경사항을 GitHub `main` 브랜치에 push해야 합니다. 공개 Google Drive 파일 다운로드 기능 자체는 실제 공개 링크가 있을 때 앱 화면에서 확인합니다.

## Setup

### 1. 저장소 준비

아래 설정을 바꾸면 다른 브랜치도 테스트할 수 있습니다. 셀은 기존 clone을 삭제하지 않고 fetch/reset하여 재실행할 수 있게 구성했습니다.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

REPO_URL = "https://github.com/JongdaeChoi/Exhibition-Data-Analysis.git"
BRANCH = "main"
REPO_DIR = Path("/content/Exhibition-Data-Analysis")
APP_DIR = REPO_DIR / "streamlit_app"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

required_files = [APP_DIR / "app.py", APP_DIR / "requirements-dev.txt"]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("GitHub에 Streamlit 코드가 없습니다. 로컬 변경을 push한 뒤 다시 실행하세요: " + ", ".join(missing))

os.chdir(APP_DIR)
print("테스트 대상:", APP_DIR)
print("커밋:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

### 2. 의존성 설치

Streamlit 실행 및 테스트에 필요한 패키지를 앱 전용 선언 파일에서 설치합니다.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dev.txt"],
    check=True,
)
print("의존성 설치 완료")

## Steps

### 3. 전체 단위 테스트

파일 적재, 원본/작업본 분리, 기본현황, 결측값, 노이즈, 특정값, 날짜 분리 테스트를 실행합니다.

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("단위 테스트가 실패했습니다. 위 오류를 확인하세요.")

### 4. 전처리 핵심 동작 확인

작은 샘플로 `df` 불변성, 결측값 대체, 특정값 변경, 날짜 요소 분리를 눈으로 확인합니다.

In [ ]:
import pandas as pd
from data.preprocessing import (
    comparison_summary,
    fill_missing_values,
    replace_selected_value,
    split_date_components,
)

df = pd.DataFrame({
    "지역": ["서울", " 서울 " , None],
    "매출": [100.0, None, 300.0],
    "등록일시": ["2026-07-15 09:30:45", "2026-07-16 17:00:00", None],
})
df_clean = df.copy(deep=True)
df_clean = fill_missing_values(df_clean, "매출", "평균값").frame
df_clean = replace_selected_value(df_clean, "지역", " 서울 ", "서울").frame
df_clean = split_date_components(df_clean, "등록일시", ["year_month", "hour"]).frame

assert pd.isna(df.loc[1, "매출"]), "원본 df가 변경되었습니다."
display(df_clean)
display(comparison_summary(df, df_clean))
print("원본 df 불변성 및 전처리 핵심 동작 확인 완료")

### 5. Streamlit 앱 실행

백그라운드에서 앱을 실행하고 `/healthz` 응답을 확인합니다.

In [ ]:
import requests

PORT = 8501
if "streamlit_process" in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.headless=true", f"--server.port={PORT}",
        "--server.enableCORS=false",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

health_url = f"http://127.0.0.1:{PORT}/_stcore/health"
for _ in range(30):
    try:
        response = requests.get(health_url, timeout=2)
        if response.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    streamlit_process.terminate()
    raise RuntimeError("Streamlit 서버가 30초 안에 시작되지 않았습니다.")

print("Streamlit health check:", response.status_code, response.text.strip())

### 6. Colab에서 앱 열기

출력되는 링크를 클릭하면 현재 Colab 런타임의 Streamlit 화면이 열립니다. Colab 프록시는 런타임이 종료되면 함께 종료됩니다.

In [ ]:
from google.colab import output
from IPython.display import HTML, display

proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
display(HTML(f'<a href="{proxy_url}" target="_blank" style="font-size:18px">Streamlit 앱 새 창에서 열기</a>'))
print(proxy_url)

## Checks

앱에서 다음 항목을 직접 확인하세요.

1. 로컬 CSV/XLSX 파일 선택 및 적재 진행 표시
2. `df`와 `df_clean` 분리 안내 및 기본현황 표
3. 결측값 처리 후 결과 요약 즉시 갱신
4. 노이즈와 Unique Value의 20개 단위 조회
5. 날짜의 년-월/월/일/시간 선택 분리
6. 전처리 이력과 CSV/XLSX 다운로드

## Next Steps

`Runtime > Run all` 실행 후에도 Streamlit 서버는 계속 유지됩니다. 테스트가 끝나면 Colab의 **런타임 연결 해제 및 삭제**를 선택하면 서버도 함께 종료됩니다. 링크가 열린 뒤에는 이 노트북의 마지막까지 실행되어도 서버가 자동 종료되지 않습니다.

### 서버를 수동으로 종료하려는 경우

일반 테스트에서는 이 작업이 필요하지 않습니다. 서버만 먼저 종료하려면 새 코드 셀을 만든 뒤 다음 코드를 직접 실행하세요.

```python
streamlit_process.terminate()
streamlit_process.wait(timeout=10)
```